# Street Light Digital Twin — Model Training

This notebook trains two models:
- **Energy Regressor** → given a light's current state, predict `energy_consumed` (kWh/hr)
- **Fault Classifier** → given a light's current state, predict `fault_probability` (0–1)

At the end, we save everything needed to serve predictions from the Node.js backend:
```
models/
  energy_model.pkl        ← trained regressor
  fault_model.pkl         ← trained classifier
  preprocessor.pkl        ← encodes/scales inputs the same way as training
  feature_columns.json    ← exact feature list in exact order (critical)
  model_metadata.json     ← thresholds, feature importances, evaluation metrics
```

## Cell 1 — Imports

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,       # regression
    classification_report, confusion_matrix,                  # classification
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score
)

warnings.filterwarnings('ignore')
np.random.seed(42)

MODELS_DIR = Path('models')
MODELS_DIR.mkdir(exist_ok=True)

print('All imports OK')
print(f'Models will be saved to: {MODELS_DIR.resolve()}')

## Cell 2 — Load and inspect the data

In [ ]:
df = pd.read_csv('synthetic_data.csv')

print(f'Shape: {df.shape}  ({df.shape[0]:,} rows × {df.shape[1]} columns)')
print(f'\nColumn types:')
print(df.dtypes)
print(f'\nFirst 3 rows:')
df.head(3)

## Cell 3 — Understand what we're predicting

Before training anything, look at the distributions of both target variables.
This tells us what kind of problem we're dealing with.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- target 1: energy_consumed ---
# only show rows where light is ON — OFF rows are always 0 and would distort the plot
energy_on = df[df['energy_consumed'] > 0]['energy_consumed']
axes[0].hist(energy_on, bins=50, color='#378ADD', edgecolor='white', linewidth=0.3)
axes[0].set_title('Energy consumed (kWh/hr)\nwhen light is ON only')
axes[0].set_xlabel('kWh per hour')
axes[0].set_ylabel('row count')
axes[0].axvline(energy_on.mean(), color='red', linestyle='--', label=f'mean={energy_on.mean():.4f}')
axes[0].legend()

# --- target 2: fault_occurred (class balance) ---
fault_counts = df['fault_occurred'].value_counts()
bars = axes[1].bar(['No fault (0)', 'Fault (1)'], fault_counts.values,
                    color=['#1D9E75', '#E24B4A'], edgecolor='white')
for bar, count in zip(bars, fault_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{count:,}\n({count/len(df)*100:.1f}%)',
                 ha='center', fontsize=10)
axes[1].set_title('Fault class balance\n(this imbalance matters for training)')
axes[1].set_ylabel('row count')

# --- fault rate by health bucket ---
df['health_bucket'] = pd.cut(df['health_score'], bins=10)
fault_by_health = df.groupby('health_bucket', observed=True)['fault_occurred'].mean()
axes[2].bar(range(len(fault_by_health)), fault_by_health.values, color='#BA7517')
axes[2].set_xticks(range(len(fault_by_health)))
axes[2].set_xticklabels([str(b) for b in fault_by_health.index], rotation=45, ha='right', fontsize=7)
axes[2].set_title('Fault rate by health score bucket\n(confirms our non-linear rule worked)')
axes[2].set_ylabel('fault rate')

plt.tight_layout()
plt.savefig('models/target_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nClass imbalance ratio: {fault_counts[0]/fault_counts[1]:.1f}:1  (no_fault : fault)')
print(f'This means we MUST use class_weight="balanced" in the classifier')
print(f'Otherwise the model will just always predict "no fault" and get {fault_counts[0]/len(df)*100:.1f}% accuracy while being useless')

## Cell 4 — Feature engineering

We select which columns are **features** (inputs to the model) vs **targets** (what we predict).
We also drop columns that would **leak** the answer — i.e. columns computed from the target.

In [ ]:
# IMPORTANT: explicitly define which columns are categorical vs numeric.
# Do NOT use df.dtypes to auto-detect — pandas may read things differently
# depending on the data, leading to subtle bugs.

# Categorical: columns with string values that need OneHotEncoding
CAT_FEATURES = ['lamp_type', 'weather']

# Numeric: columns with number values that get StandardScaled
NUM_FEATURES = ['rated_power', 'efficiency', 'hour', 'is_night',
                'brightness', 'ambient_light', 'health_score']

FEATURE_COLS = CAT_FEATURES + NUM_FEATURES

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATURES),
    ('num', StandardScaler(), NUM_FEATURES),
], remainder='drop')

X_raw = df[FEATURE_COLS]
preprocessor.fit(X_raw)
X = preprocessor.transform(X_raw)

cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(CAT_FEATURES)
final_feature_names = list(cat_feature_names) + NUM_FEATURES

print(f'Features before encoding: {len(FEATURE_COLS)}')
print(f'Features after encoding:  {X.shape[1]}')
print(f'\nFinal feature names:')
for i, name in enumerate(final_feature_names):
    print(f'  {i+1:2d}. {name}')

feature_info = {
    'raw_feature_columns':   FEATURE_COLS,
    'categorical_features':  CAT_FEATURES,
    'numeric_features':      NUM_FEATURES,
    'final_feature_names':   list(final_feature_names),
    'cat_values': {col: sorted(df[col].unique().tolist()) for col in CAT_FEATURES}
}
with open(MODELS_DIR / 'feature_columns.json', 'w') as f:
    json.dump(feature_info, f, indent=2)

print(f'\nSaved feature_columns.json')
print(f'Categorical value sets: {feature_info["cat_values"]}')

## Cell 5 — Build the preprocessor

The preprocessor handles encoding + scaling in one reusable object.
**Saving this alongside the model is critical** — when your backend receives a prediction request,
it must transform the input the exact same way as training data was transformed.
If you encode `weather` differently at prediction time, the model gets garbage input.

In [ ]:
# ColumnTransformer applies different transforms to different column types:
#   - categorical columns  → OneHotEncoder (converts 'rainy' to [0,0,1,0,0])
#   - numeric columns      → StandardScaler (zero mean, unit variance)
#
# Note: Random Forests don't actually need scaling (they're tree-based, not distance-based).
# We still include StandardScaler because:
#   1. It makes feature importances comparable across different scales
#   2. If you ever swap in a different model (SVM, neural net) later, preprocessing stays the same
#   3. The Flask prediction endpoint just passes data through the preprocessor — one less thing to worry about

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATURES),
    ('num', StandardScaler(), NUM_FEATURES),
], remainder='drop')

# fit the preprocessor on the full dataset (not just train split)
# this ensures it has seen all possible weather/lamp_type values
X_raw = df[FEATURE_COLS]
preprocessor.fit(X_raw)
X = preprocessor.transform(X_raw)

# get the final feature names after encoding
# e.g. 'weather' becomes 'weather_clear', 'weather_rainy', etc.
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(CAT_FEATURES)
final_feature_names = list(cat_feature_names) + NUM_FEATURES

print(f'Features before encoding: {len(FEATURE_COLS)}')
print(f'Features after encoding:  {X.shape[1]}')
print(f'\nFinal feature names:')
for i, name in enumerate(final_feature_names):
    print(f'  {i+1:2d}. {name}')

# save feature columns info — backend needs this to build input correctly
feature_info = {
    'raw_feature_columns':   FEATURE_COLS,
    'categorical_features':  CAT_FEATURES,
    'numeric_features':      NUM_FEATURES,
    'final_feature_names':   list(final_feature_names),
    'cat_values': {
        col: sorted(df[col].unique().tolist()) for col in CAT_FEATURES
    }
}
with open(MODELS_DIR / 'feature_columns.json', 'w') as f:
    json.dump(feature_info, f, indent=2)

print(f'\nSaved feature_columns.json')
print(f'Categorical value sets: {feature_info["cat_values"]}')

## Cell 6 — Train/test split

We split by **day**, not randomly. This is important.

Random splitting would leak future health states into training.
Example: if day 150 and day 151 of the same light land in train and test,
the model sees the health score on day 150 during training, which is nearly
identical to day 151 in the test set. That inflates test accuracy.

Time-based split: train on days 0–143 (first 80%), test on days 144–179 (last 20%).
The model has never seen the test period at all during training.

In [ ]:
SPLIT_DAY = int(180 * 0.8)   # day 144

train_mask = df['day'] < SPLIT_DAY
test_mask  = df['day'] >= SPLIT_DAY

X_train = preprocessor.transform(df.loc[train_mask, FEATURE_COLS])
X_test  = preprocessor.transform(df.loc[test_mask,  FEATURE_COLS])

y_energy_train = df.loc[train_mask, TARGET_ENERGY].values
y_energy_test  = df.loc[test_mask,  TARGET_ENERGY].values

y_fault_train  = df.loc[train_mask, TARGET_FAULT].values
y_fault_test   = df.loc[test_mask,  TARGET_FAULT].values

print(f'Split day: {SPLIT_DAY}  (train = days 0–{SPLIT_DAY-1}, test = days {SPLIT_DAY}–179)')
print(f'\nTrain rows: {X_train.shape[0]:,}  ({X_train.shape[0]/len(df)*100:.0f}% of data)')
print(f'Test rows:  {X_test.shape[0]:,}   ({X_test.shape[0]/len(df)*100:.0f}% of data)')
print(f'\nFault rate in train: {y_fault_train.mean():.4f}')
print(f'Fault rate in test:  {y_fault_test.mean():.4f}')
print('(These should be similar — confirms no accidental data leakage from the split)')

## Cell 7 — Train the Energy Regressor

Predicts `energy_consumed` (kWh/hr) given current light state.
Used in the what-if engine to predict energy savings from brightness changes.

In [ ]:
print('Training Energy Regressor...')
print('(this takes ~30–60 seconds)')

energy_model = RandomForestRegressor(
    n_estimators=100,       # 100 trees — good balance of accuracy vs speed
    max_depth=20,           # allow deep trees; energy has a clean relationship with features
    min_samples_leaf=5,     # each leaf needs at least 5 samples — prevents extreme overfitting
    n_jobs=-1,              # use all CPU cores
    random_state=42
)

energy_model.fit(X_train, y_energy_train)

# --- evaluate ---
y_pred_energy = energy_model.predict(X_test)

mae  = mean_absolute_error(y_energy_test, y_pred_energy)
rmse = np.sqrt(mean_squared_error(y_energy_test, y_pred_energy))
r2   = r2_score(y_energy_test, y_pred_energy)

# mean absolute percentage error (only on rows where light is ON)
mask_on = y_energy_test > 0
mape = np.mean(np.abs((y_energy_test[mask_on] - y_pred_energy[mask_on]) / y_energy_test[mask_on])) * 100

print(f'\n── Energy Regressor Results ──────────────────')
print(f'  R² score:  {r2:.4f}   (1.0 = perfect, >0.95 is excellent)')
print(f'  MAE:       {mae:.6f} kWh/hr')
print(f'  RMSE:      {rmse:.6f} kWh/hr')
print(f'  MAPE:      {mape:.2f}%  (when light is ON)')

# save the model
joblib.dump(energy_model, MODELS_DIR / 'energy_model.pkl')
print(f'\nSaved energy_model.pkl')

## Cell 8 — Energy model: visualise predictions vs actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- scatter: predicted vs actual (sample 3000 points so it's readable) ---
idx = np.random.choice(len(y_energy_test), size=3000, replace=False)
axes[0].scatter(y_energy_test[idx], y_pred_energy[idx],
                alpha=0.3, s=8, color='#378ADD')
lim = max(y_energy_test.max(), y_pred_energy.max()) * 1.05
axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1, label='perfect prediction')
axes[0].set_xlabel('Actual energy (kWh/hr)')
axes[0].set_ylabel('Predicted energy (kWh/hr)')
axes[0].set_title(f'Energy: Predicted vs Actual\nR²={r2:.4f}, MAPE={mape:.1f}%')
axes[0].legend()

# --- feature importances ---
importances = energy_model.feature_importances_
fi_df = pd.DataFrame({'feature': final_feature_names, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=True).tail(12)  # top 12

axes[1].barh(fi_df['feature'], fi_df['importance'], color='#1D9E75')
axes[1].set_xlabel('Feature importance')
axes[1].set_title('Energy model: top feature importances')

plt.tight_layout()
plt.savefig('models/energy_model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print('What you should see:')
print('  Scatter: tight cluster along the red diagonal → model predicts well')
print('  Importances: brightness, rated_power, efficiency/lamp_type should dominate')
print('  (This confirms the model learned the physics, not noise)')

## Cell 9 — Train the Fault Classifier

Predicts probability of a fault occurring in a given hour.
Used in the what-if engine to predict fault risk under different operating conditions.

Key difference from the regressor: `class_weight='balanced'` is essential here
because only ~3% of rows are actual faults.

In [ ]:
print('Training Fault Classifier...')
print('(this takes ~30–60 seconds)')

fault_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,             # slightly shallower than regressor — fault signal is noisier
    min_samples_leaf=10,      # more conservative — fault events are rare and noisy
    class_weight='balanced',  # THIS IS THE KEY SETTING FOR IMBALANCED DATA
                              # internally up-weights the minority class (faults)
                              # so the model actually tries to predict them
    n_jobs=-1,
    random_state=42
)

fault_model.fit(X_train, y_fault_train)

# --- evaluate ---
y_pred_fault       = fault_model.predict(X_test)           # binary 0/1 prediction
y_pred_fault_proba = fault_model.predict_proba(X_test)[:, 1]  # probability of fault

roc_auc = roc_auc_score(y_fault_test, y_pred_fault_proba)
avg_prec = average_precision_score(y_fault_test, y_pred_fault_proba)

print(f'\n── Fault Classifier Results ──────────────────')
print(f'  ROC-AUC:           {roc_auc:.4f}  (1.0 = perfect, >0.85 is good for imbalanced data)')
print(f'  Avg Precision:     {avg_prec:.4f}  (summary of precision-recall curve)')
print(f'\n  Detailed report (at default 0.5 threshold):')
print(classification_report(y_fault_test, y_pred_fault,
                              target_names=['No Fault', 'Fault']))

# save model
joblib.dump(fault_model, MODELS_DIR / 'fault_model.pkl')
print('Saved fault_model.pkl')

## Cell 10 — Fault model: visualise performance

For an imbalanced classifier, accuracy is a terrible metric.
We use ROC curve and Precision-Recall curve instead.

**ROC curve**: how well the model separates faults from non-faults across all thresholds.
**Precision-Recall curve**: more informative for imbalanced data — shows the tradeoff
between catching more faults (recall) vs generating false alarms (precision).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- ROC curve ---
fpr, tpr, roc_thresholds = roc_curve(y_fault_test, y_pred_fault_proba)
axes[0].plot(fpr, tpr, color='#378ADD', lw=2, label=f'ROC curve (AUC={roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='random classifier')
axes[0].set_xlabel('False Positive Rate (false alarms)')
axes[0].set_ylabel('True Positive Rate (faults caught)')
axes[0].set_title('ROC Curve — Fault Classifier')
axes[0].legend()

# --- Precision-Recall curve ---
prec, rec, pr_thresholds = precision_recall_curve(y_fault_test, y_pred_fault_proba)
axes[1].plot(rec, prec, color='#D85A30', lw=2, label=f'AP={avg_prec:.3f}')
axes[1].axhline(y=y_fault_test.mean(), color='k', linestyle='--', lw=1,
                 label=f'baseline ({y_fault_test.mean():.3f})')
axes[1].set_xlabel('Recall (fraction of faults caught)')
axes[1].set_ylabel('Precision (fraction of alerts that are real)')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

# --- Confusion matrix ---
cm = confusion_matrix(y_fault_test, y_pred_fault)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
             xticklabels=['Pred: No Fault', 'Pred: Fault'],
             yticklabels=['True: No Fault', 'True: Fault'])
axes[2].set_title('Confusion Matrix\n(at default 0.5 threshold)')

plt.tight_layout()
plt.savefig('models/fault_model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

# --- find the optimal threshold ---
# default threshold (0.5) is arbitrary for imbalanced data.
# We pick the threshold that maximises F1 score on the test set.
f1_scores = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-9)
best_idx = np.argmax(f1_scores)
optimal_threshold = pr_thresholds[best_idx]
print(f'\nOptimal fault probability threshold: {optimal_threshold:.4f}')
print(f'At this threshold:')
print(f'  Precision: {prec[best_idx]:.3f}  (of lights flagged as faulty, {prec[best_idx]*100:.1f}% actually fault)')
print(f'  Recall:    {rec[best_idx]:.3f}  (of actual faults, we catch {rec[best_idx]*100:.1f}%)')
print(f'  F1:        {f1_scores[best_idx]:.3f}')
print(f'\nThis threshold is saved in model_metadata.json and used by the Flask API')

## Cell 11 — Fault model: feature importances

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- energy model importances ---
fi_energy = pd.DataFrame({
    'feature': final_feature_names,
    'importance': energy_model.feature_importances_
}).sort_values('importance', ascending=True)

axes[0].barh(fi_energy['feature'], fi_energy['importance'], color='#378ADD')
axes[0].set_title('Energy model\nfeature importances')
axes[0].set_xlabel('Importance')

# --- fault model importances ---
fi_fault = pd.DataFrame({
    'feature': final_feature_names,
    'importance': fault_model.feature_importances_
}).sort_values('importance', ascending=True)

axes[1].barh(fi_fault['feature'], fi_fault['importance'], color='#D85A30')
axes[1].set_title('Fault model\nfeature importances')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('models/feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()

print('What you should see:')
print('  Energy model: brightness, rated_power dominate (matches our formula)')
print('  Fault model:  health_score dominates, then weather, lamp_type')
print('  This separation confirms each model learned the right physical relationships')

## Cell 12 — Save the preprocessor and all metadata

This is the most important cell for the backend.
Everything saved here is loaded by the Flask prediction server.

In [ ]:
# save the preprocessor — MUST be saved alongside the models
joblib.dump(preprocessor, MODELS_DIR / 'preprocessor.pkl')
print('Saved preprocessor.pkl')

# save all metadata the Flask server and backend need
metadata = {
    # how to use the models
    'fault_threshold': float(optimal_threshold),
    'raw_feature_columns': FEATURE_COLS,
    'categorical_features': CAT_FEATURES,
    'numeric_features': NUM_FEATURES,
    'final_feature_names': list(final_feature_names),

    # allowed values for categorical inputs — Flask validates against these
    'allowed_values': {
        col: sorted(df[col].unique().tolist()) for col in CAT_FEATURES
    },

    # numeric input ranges — Flask validates against these
    'input_ranges': {
        col: {'min': float(df[col].min()), 'max': float(df[col].max())}
        for col in NUM_FEATURES
    },

    # evaluation metrics — useful to display in dashboard
    'energy_model_metrics': {
        'r2': float(r2),
        'mae': float(mae),
        'rmse': float(rmse),
        'mape_pct': float(mape)
    },
    'fault_model_metrics': {
        'roc_auc': float(roc_auc),
        'avg_precision': float(avg_prec),
        'precision_at_threshold': float(prec[best_idx]),
        'recall_at_threshold': float(rec[best_idx]),
        'f1_at_threshold': float(f1_scores[best_idx])
    },

    # feature importances — useful for dashboard explainability panel
    'energy_feature_importances': dict(zip(
        final_feature_names,
        [float(x) for x in energy_model.feature_importances_]
    )),
    'fault_feature_importances': dict(zip(
        final_feature_names,
        [float(x) for x in fault_model.feature_importances_]
    )),

    # training config
    'train_days': f'0–{SPLIT_DAY-1}',
    'test_days':  f'{SPLIT_DAY}–179',
    'train_rows': int(X_train.shape[0]),
    'test_rows':  int(X_test.shape[0]),
    'total_lights': 50,
    'simulation_days': 180,
}

with open(MODELS_DIR / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Saved model_metadata.json')

print('\n── Summary of saved files ──────────────────────────────')
for p in sorted(MODELS_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name:<35} {size_kb:>8.1f} KB')

## Cell 13 — Test a prediction end-to-end

Simulate exactly what the Flask API will do when it receives a request from Express.
This is your sanity check that the save/load pipeline works correctly.

In [ ]:
# load everything back from disk — same as what Flask will do on startup
loaded_energy_model = joblib.load(MODELS_DIR / 'energy_model.pkl')
loaded_fault_model  = joblib.load(MODELS_DIR / 'fault_model.pkl')
loaded_preprocessor = joblib.load(MODELS_DIR / 'preprocessor.pkl')
with open(MODELS_DIR / 'model_metadata.json') as f:
    loaded_meta = json.load(f)

print('All files loaded successfully from disk')

# --- scenario A: LED light, good health, clear night ---
def predict(light_state: dict) -> dict:
    """
    This function will become the core of your Flask /predict endpoint.
    Input:  dict with the raw feature values (same keys as FEATURE_COLS)
    Output: dict with energy prediction + fault probability
    """
    row = pd.DataFrame([light_state])[loaded_meta['raw_feature_columns']]
    X   = loaded_preprocessor.transform(row)

    energy      = float(loaded_energy_model.predict(X)[0])
    fault_prob  = float(loaded_fault_model.predict_proba(X)[0][1])
    fault_alert = fault_prob >= loaded_meta['fault_threshold']

    return {
        'predicted_energy_kwh':   round(energy, 6),
        'fault_probability':      round(fault_prob, 4),
        'fault_alert':            fault_alert,
        'fault_threshold_used':   loaded_meta['fault_threshold'],
    }


# test case A: healthy LED, clear weather, night, 80% brightness
scenario_a = {
    'lamp_type': 'LED', 'rated_power': 100, 'efficiency': 1.0,
    'hour': 22, 'is_night': 1, 'brightness': 80.0,
    'ambient_light': 5.0, 'weather': 'clear', 'health_score': 90.0
}
result_a = predict(scenario_a)
print(f'\nScenario A — Healthy LED, clear, 80% brightness at hour 22:')
print(f'  Predicted energy:    {result_a["predicted_energy_kwh"]:.6f} kWh/hr')
print(f'  Fault probability:   {result_a["fault_probability"]:.4f}  ({result_a["fault_probability"]*100:.2f}%)')
print(f'  Fault alert:         {result_a["fault_alert"]}')

# test case B: same light, stormy, low health
scenario_b = {**scenario_a, 'weather': 'stormy', 'health_score': 15.0}
result_b = predict(scenario_b)
print(f'\nScenario B — Same LED, stormy weather, health=15:')
print(f'  Predicted energy:    {result_b["predicted_energy_kwh"]:.6f} kWh/hr')
print(f'  Fault probability:   {result_b["fault_probability"]:.4f}  ({result_b["fault_probability"]*100:.2f}%)')
print(f'  Fault alert:         {result_b["fault_alert"]}')

# test case C: what-if scenario — reduce brightness to 50%
scenario_c = {**scenario_a, 'brightness': 50.0}
result_c = predict(scenario_c)
energy_saving_pct = (result_a['predicted_energy_kwh'] - result_c['predicted_energy_kwh']) / result_a['predicted_energy_kwh'] * 100
print(f'\nScenario C — What-if: same LED at 50% brightness (vs 80%):')
print(f'  Predicted energy:    {result_c["predicted_energy_kwh"]:.6f} kWh/hr')
print(f'  Energy saving:       {energy_saving_pct:.1f}%  vs baseline')
print(f'  Fault probability:   {result_c["fault_probability"]:.4f}  (dimmer = slightly less thermal stress)')

## Cell 14 — Write the Flask prediction server

This writes `flask_model_api.py` to disk — the file you run alongside your Express backend.
Express calls this server at `http://localhost:5001/predict`.
Run it with: `python flask_model_api.py`

In [ ]:
flask_code = '''
"""
Flask Model API — Street Light Digital Twin
===========================================
Run this server alongside your Express backend.
Express calls POST /predict with a light state JSON.
This server returns energy prediction + fault probability.

Start with:  python flask_model_api.py
Runs on:     http://localhost:5001
"""

import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)   # allow Express (different port) to call this

# ── load models on startup (once, not per request) ───────────────────────────
MODELS_DIR = Path("models")

print("Loading models...")
energy_model = joblib.load(MODELS_DIR / "energy_model.pkl")
fault_model  = joblib.load(MODELS_DIR / "fault_model.pkl")
preprocessor = joblib.load(MODELS_DIR / "preprocessor.pkl")

with open(MODELS_DIR / "model_metadata.json") as f:
    meta = json.load(f)

FEATURE_COLS     = meta["raw_feature_columns"]
FAULT_THRESHOLD  = meta["fault_threshold"]
ALLOWED_VALUES   = meta["allowed_values"]
INPUT_RANGES     = meta["input_ranges"]
print(f"Models loaded. Fault threshold: {FAULT_THRESHOLD:.4f}")


def validate_input(data: dict) -> tuple[bool, str]:
    """Check incoming request has all required fields with valid values."""
    for col in FEATURE_COLS:
        if col not in data:
            return False, f"Missing field: {col}"

    for col, allowed in ALLOWED_VALUES.items():
        if data[col] not in allowed:
            return False, f"Invalid value for {col}: {data[col]}. Allowed: {allowed}"

    for col, rng in INPUT_RANGES.items():
        val = data.get(col)
        if val is not None and not (rng["min"] - 1 <= float(val) <= rng["max"] + 1):
            return False, f"{col}={val} is outside expected range [{rng[\"min\"]}, {rng[\"max\"]}]"

    return True, ""


@app.route("/health", methods=["GET"])
def health():
    """Express calls this on startup to check the model server is up."""
    return jsonify({"status": "ok", "models_loaded": True,
                    "fault_threshold": FAULT_THRESHOLD})


@app.route("/predict", methods=["POST"])
def predict():
    """Single-light prediction."""
    data = request.get_json()
    if not data:
        return jsonify({"error": "No JSON body"}), 400

    ok, err = validate_input(data)
    if not ok:
        return jsonify({"error": err}), 422

    row  = pd.DataFrame([data])[FEATURE_COLS]
    X    = preprocessor.transform(row)

    energy     = float(energy_model.predict(X)[0])
    fault_prob = float(fault_model.predict_proba(X)[0][1])

    return jsonify({
        "predicted_energy_kwh": round(max(0.0, energy), 6),
        "fault_probability":    round(fault_prob, 4),
        "fault_alert":          fault_prob >= FAULT_THRESHOLD,
        "fault_threshold_used": FAULT_THRESHOLD,
    })


@app.route("/predict/batch", methods=["POST"])
def predict_batch():
    """
    Predict for multiple lights at once.
    The twin engine calls this every tick with all 50 lights
    instead of making 50 individual /predict calls.
    Much faster.
    Input:  { "lights": [ {light_state}, {light_state}, ... ] }
    Output: { "predictions": [ {result}, {result}, ... ] }
    """
    data = request.get_json()
    if not data or "lights" not in data:
        return jsonify({"error": 'Expected {"lights": [...]}'}), 400

    lights = data["lights"]
    if not lights:
        return jsonify({"predictions": []})

    df_input = pd.DataFrame(lights)[FEATURE_COLS]
    X        = preprocessor.transform(df_input)

    energies     = energy_model.predict(X)
    fault_probas = fault_model.predict_proba(X)[:, 1]

    predictions = [
        {
            "predicted_energy_kwh": round(float(max(0.0, e)), 6),
            "fault_probability":    round(float(fp), 4),
            "fault_alert":          float(fp) >= FAULT_THRESHOLD,
        }
        for e, fp in zip(energies, fault_probas)
    ]

    return jsonify({"predictions": predictions})


@app.route("/metadata", methods=["GET"])
def metadata():
    """Return model metadata — dashboard can show model performance stats."""
    return jsonify(meta)


if __name__ == "__main__":
    print("Starting Flask model server on http://localhost:5001")
    print("Endpoints:")
    print("  GET  /health          — liveness check")
    print("  POST /predict         — single light prediction")
    print("  POST /predict/batch   — batch prediction (all 50 lights at once)")
    print("  GET  /metadata        — model metrics and config")
    app.run(host="0.0.0.0", port=5001, debug=False)
'''

with open('flask_model_api.py', 'w') as f:
    f.write(flask_code.strip())

print('Written: flask_model_api.py')
print('\nTo start the model server:')
print('  pip install flask flask-cors')
print('  python flask_model_api.py')
print('\nTo test it from terminal:')
print('''  curl -X POST http://localhost:5001/predict \\''')
print('''       -H "Content-Type: application/json" \\''')
print('''       -d '{"lamp_type":"LED","rated_power":100,"efficiency":1.0,''')
print('''            "hour":22,"is_night":1,"brightness":80,''')
print('''            "ambient_light":5,"weather":"clear","health_score":90}' ''')

## Cell 15 — Final summary

In [ ]:
print('=' * 58)
print('TRAINING COMPLETE — EVERYTHING SAVED')
print('=' * 58)

print(f'''
Models trained and saved to models/
─────────────────────────────────────────────────────
  energy_model.pkl      RandomForestRegressor
                        R²={r2:.4f}, MAPE={mape:.1f}%

  fault_model.pkl       RandomForestClassifier
                        ROC-AUC={roc_auc:.4f}
                        Optimal threshold: {optimal_threshold:.4f}

  preprocessor.pkl      ColumnTransformer
                        OneHotEncoder (weather, lamp_type)
                        StandardScaler (numeric features)

  model_metadata.json   Thresholds, metrics, feature info,
                        allowed values, input ranges

  feature_columns.json  Raw and encoded feature names

Flask server written to:
  flask_model_api.py

Next steps:
  1. pip install flask flask-cors
  2. python flask_model_api.py        (runs on port 5001)
  3. Test: GET http://localhost:5001/health
  4. In Express, use axios to POST to http://localhost:5001/predict/batch
     with all 50 lights every twin engine tick
''')

print('Feature order the Flask API expects (9 raw features):')
for i, col in enumerate(FEATURE_COLS):
    print(f'  {i+1}. {col}')